In [2]:
import pandas as pd
import random,glob

# Tirage des échantillons de semaine
- 600 formes verbales par jour, 7 jours par semaines
    - 4200 formes
- 52 semaines par an pendant 8 ans
    - 416 échantillons de 4200 formes

In [9]:
chunk=4200
nbChunks=52*8

In [4]:
fVerbes3='/Users/gilles/ownCloud/Recherche/Boye/HDR/Data/Samples/MGC-171229-Verbes3.pkl'
repDesintegrationSamples="/Users/gilles/ownCloud/Recherche/Boye/HDR/Data/Desintegration/Desintegration-Samples/"

### Lecture de la base de verbes

In [3]:
dfLex=pd.read_pickle(fVerbes3)

In [4]:
del dfLex["ext"]
del dfLex["cs"]
del dfLex["ms"]
del dfLex["vs"]
del dfLex["prob"]

#### Standardisation de la phonologie en notation API

In [5]:
# traduire SAMPA-BDLex en API

def sampa2api(sampa):
    if isinstance(sampa,str):
        api=sampa.decode("utf8")
    else:
        api=sampa
    api=api.replace(u'n"',u'n') 
    api=api.replace(u't"',u't') 
    api=api.replace(u'z"',u'z') 
    api=api.replace(u'R"',u'ʁ') 
    api=api.replace(u'p"',u'p') 
    api=api.replace(u'S',u'ʃ') 
    api=api.replace(u'Z',u'ʒ')
    api=api.replace(u'N',u'ŋ')
    api=api.replace(u'J',u'ɲ')
    api=api.replace(u'r',u'ʁ') 
    api=api.replace(u'H',u'ɥ')
    api=api.replace(u'E',u'ɛ')
    api=api.replace(u'2',u'ø')
    api=api.replace(u'9',u'œ')
    api=api.replace(u'6',u'ə')
    api=api.replace(u'O',u'ɔ')
    api=api.replace(u'è',u'e')   
    api=api.replace(u'ò',u'o')    
    api=api.replace(u'â',u'ɑ̃')   
    api=api.replace(u'ê',u'ɛ̃')   
    api=api.replace(u'û',u'œ̃')  
    api=api.replace(u'ô',u'ɔ̃')       
    api=api.replace(u'@',u'ə')
    api=api.replace(u'R',u'ʁ') 
    return api

In [6]:
dfLex["phono"]=dfLex["phono"].apply(sampa2api)

#### Définition de la fréquence cumulée

In [7]:
dfLex["freqcum"]= (dfLex["freq"].cumsum()*1000).astype(int)
freqMax=dfLex["freqcum"].max()

In [8]:
dfLex

,ortho,phono,lexeme,freq,case,freqcum
40,a,a,avoir,7.167735e+11,pi3S,716773482510576
45,abaissa,abɛsa,abaisser,2.640001e+08,ai3S,717037482636240
46,abaissai,abɛsɛ,abaisser,1.700001e+07,ai1S,717054482650551
47,abaissaient,abɛsɛ,abaisser,4.100005e+07,ii3P,717095482704558
48,abaissais,abɛsɛ,abaisser,4.933548e+01,ii1S,717095482753893
49,abaissais,abɛsɛ,abaisser,4.954692e-01,ii2S,717095482754389
50,abaissait,abɛsɛ,abaisser,2.520003e+08,ii3S,717347483019956
52,abaissant,abɛsɑ̃,abaisser,2.510001e+08,pP,717598483083327
56,abaissas,abɛsa,abaisser,2.565900e-01,ai2S,717598483083584
57,abaissasse,abɛsas,abaisser,3.093462e-01,is1S,717598483083893


### Définition des tirages

In [10]:
%%time
tirage=[]
for n in range(nbChunks*chunk):
    tirage.append(random.randrange(freqMax))

CPU times: user 3.77 s, sys: 240 ms, total: 4.01 s
Wall time: 3.38 s


In [11]:
dfLex["tir1"]=0

In [12]:
def tirage2triage(tirage,cumul=True):    
    triage=sorted(tirage)
    freqTop=0
    indexMin=0
    tirs={}

    for num,tir in enumerate(triage[:]):
        if tir > freqTop:
            indexMin=dfLex[dfLex["freqcum"]>=tir][0:1].index.astype(int)[0]
            freqTop=dfLex.ix[indexMin,'freqcum']
            tirs[indexMin]=0
        tirs[indexMin] += 1
        if num%500000==0:
            print num,
    print
    if cumul:
        for indexNum in tirs:
            dfLex.ix[indexNum,'tir1']+=tirs[indexNum]
    else:
        dfLex['tir1']=0
        for indexNum in tirs:
            dfLex.ix[indexNum,'tir1']=tirs[indexNum]
        

In [13]:
for nbChunk in range(nbChunks):
    %time tirage2triage(tirage[nbChunk*chunk:(nbChunk+1)*chunk],cumul=False)
    print
    print "nbChunk",nbChunk
    print dfLex[dfLex["tir1"]!=0]["tir1"].sum()
    print dfLex[dfLex["tir1"]!=0]["tir1"].count()
    dfSample=dfLex[dfLex["tir1"]!=0]["lexeme case phono tir1".split(" ")]
    dfSample.to_csv(path_or_buf=repDesintegrationSamples+"DS-%03d-%d.csv"%(nbChunk,chunk),sep="\t",encoding="utf8")
    display(dfSample)
    # with open("/Users/gilles/Box Sync/2015-Data/"+tiragePrefix+"%02d"%(nTile)+echantillon+'-Tirage.pkl', 'wb') as output:
    #     pickle.dump(lexique, output, pickle.HIGHEST_PROTOCOL)

/Users/gilles/opt/anaconda3/envs/python2/lib/python2.7/site-packages/ipykernel_launcher.py:10: DeprecationWarning: 
.ix is deprecated. Please use
.loc for label based indexing or
.iloc for positional indexing

See the documentation here:
http://pandas.pydata.org/pandas-docs/stable/indexing.html#ix-indexer-is-deprecated
  # Remove the CWD from sys.path while we load stuff.


0


/Users/gilles/opt/anaconda3/envs/python2/lib/python2.7/site-packages/ipykernel_launcher.py:22: DeprecationWarning: 
.ix is deprecated. Please use
.loc for label based indexing or
.iloc for positional indexing

See the documentation here:
http://pandas.pydata.org/pandas-docs/stable/indexing.html#ix-indexer-is-deprecated


CPU times: user 1min 10s, sys: 17.5 s, total: 1min 28s
Wall time: 31.5 s

nbChunk 0
4200
1896


,lexeme,case,phono,tir1
40,avoir,pi3S,a,95
132,abandonner,inf,abɑ̃dɔne,2
164,abandonner,ppMS,abɑ̃dɔne,1
243,abattre,pi3S,aba,1
1441,abouter,pI1P,abutɔ̃,1
1943,absenter,pi1S,absɑ̃t,1
2766,accabler,ppMP,akable,1
2887,accentuer,ii3S,aksɑ̃tyɛ,1
2950,accepter,ii3S,aksɛptɛ,1
2971,accepter,inf,aksɛpte,1


...

,lexeme,case,phono,tir1
40,avoir,pi3S,a,78
102,abaisser,ppFS,abɛse,1
130,abandonner,pi3P,abɑ̃dɔn,1
132,abandonner,inf,abɑ̃dɔne,1
145,abandonner,pi2S,abɑ̃dɔn,1
272,abattre,pi2P,abate,1
2334,abuser,pI2S,abyz,1
2712,accabler,ii3P,akablɛ,1
2950,accepter,ii3S,aksɛptɛ,3
2967,accepter,pi3S,aksɛpt,1


0
CPU times: user 1min 17s, sys: 22.5 s, total: 1min 39s
Wall time: 30.2 s

nbChunk 414
4200
1967


,lexeme,case,phono,tir1
40,avoir,pi3S,a,87
164,abandonner,ppMS,abɑ̃dɔne,1
256,abattre,ii1S,abatɛ,1
305,abattre,ppMS,abaty,1
1121,aborder,ai3S,abɔʁda,1
1434,aboutir,ii1P,abutisjɔ̃,1
1468,aboyer,inf,abwaje,1
1673,abriter,ii3S,abʁitɛ,1
2100,absorber,ppMS,absɔʁbe,1
2339,abuser,inf,abyze,1


0
CPU times: user 1min 16s, sys: 22 s, total: 1min 38s
Wall time: 29.8 s

nbChunk 415
4200
1925


,lexeme,case,phono,tir1
40,avoir,pi3S,a,89
113,abandonner,ii3S,abɑ̃dɔnɛ,1
166,abandonner,ppFS,abɑ̃dɔne,1
310,abattre,ppFP,abaty,1
1147,aborder,inf,abɔʁde,1
1468,aboyer,inf,abwaje,1
1668,abriter,ai3S,abʁita,1
2138,abstenir,pi2P,abstəne,1
2155,abstenir,ppFS,abstəny,1
2332,abuser,pi1S,abyz,1


# Longitudinales avec désintégration
- Pour chaque échantillon *i*, on propage les tirages à *i+1* en les divisant par *pDivision* (par exemple 2) pour tenir compte de la désintégration.
- La désintégration est complète quand un tirage devient inférieur au seuil *vDesintegration*. La forme en question disparaît de l'échantillon.  
Le seuil est : $${vDesintegration}=\frac{1}{{pDivision}^{cDesintegration}}$$

Autrement dit pour un élément tiré une fois dans un échantillon et plus par la suite, sa trace disparaît après *cDesintegration* tirages.

In [108]:
pDivision=2
cDesintegration=7
vDesintegration=float(1)/pDivision**cDesintegration
vDesintegration

0.0078125

In [5]:
lSamples=glob.glob(repDesintegrationSamples+"*.csv")
repDesintegrationLongitudinales='/Users/gilles/sDrive/Recherche/Boye/HDR/Data/Desintegration/Desintegration-Longitudinales/'

In [110]:
df0=pd.read_csv(lSamples[0],sep="\t",encoding="utf8",index_col=[0,1,2,3])
df0["tir2"]=(df0["tir1"]/pDivision).astype(float)
df0.to_csv(path_or_buf=repDesintegrationLongitudinales+"DSL-%03d-%d-%d-%d.csv"%(0,chunk,pDivision,cDesintegration),sep="\t",encoding="utf8")
df0=df0.rename(columns={"tir1":"tir0", "tir2":"tir1"})
df0.head()

,,,,tir0,tir1
,lexeme,case,phono,,
40,avoir,pi3S,a,95,47.5
132,abandonner,inf,abɑ̃dɔne,2,1.0
164,abandonner,ppMS,abɑ̃dɔne,1,0.5
243,abattre,pi3S,aba,1,0.5
1441,abouter,pI1P,abutɔ̃,1,0.5


In [111]:
for i in range(1,len(lSamples[:])):
    df1=pd.read_csv(lSamples[i],sep="\t",encoding="utf8",index_col=[0,1,2,3])
    df1["tir2"]=df1["tir1"]
    df1=df1.add(df0,fill_value=0)
    # display(df1.head())
    df2=df1[df1.tir1>=vDesintegration][["tir1","tir2"]]
    df2["tir2"]=df2["tir1"]/pDivision
    display(df2.head())
    df2.to_csv(path_or_buf=repDesintegrationLongitudinales+"DSL-%03d-%d-%d-%d.csv"%(i,chunk,pDivision,cDesintegration),sep="\t",encoding="utf8")
    df0=df2
    df0=df0.rename(columns={"tir1":"tir0", "tir2":"tir1"})

,,,,tir1,tir2
,lexeme,case,phono,,
40,avoir,pi3S,a,148.5,74.25
132,abandonner,inf,abɑ̃dɔne,1.0,0.50
145,abandonner,pi2S,abɑ̃dɔn,1.0,0.50
164,abandonner,ppMS,abɑ̃dɔne,1.5,0.75
243,abattre,pi3S,aba,0.5,0.25


,,,,tir1,tir2
,lexeme,case,phono,,
40,avoir,pi3S,a,156.25,78.125
128,abandonner,pi3S,abɑ̃dɔn,2.00,1.000
132,abandonner,inf,abɑ̃dɔne,2.50,1.250
145,abandonner,pi2S,abɑ̃dɔn,0.50,0.250
164,abandonner,ppMS,abɑ̃dɔne,0.75,0.375


,,,,tir1,tir2
,lexeme,case,phono,,
40,avoir,pi3S,a,152.125,76.0625
73,abaisser,inf,abɛse,1.000,0.5000
128,abandonner,pi3S,abɑ̃dɔn,1.000,0.5000
132,abandonner,inf,abɑ̃dɔne,3.250,1.6250
145,abandonner,pi2S,abɑ̃dɔn,0.250,0.1250


,,,,tir1,tir2
,lexeme,case,phono,,
40,avoir,pi3S,a,160.0625,80.03125
73,abaisser,inf,abɛse,0.5000,0.25000
128,abandonner,pi3S,abɑ̃dɔn,1.5000,0.75000
132,abandonner,inf,abɑ̃dɔne,4.6250,2.31250
133,abandonner,fi3S,abɑ̃dɔnəʁa,1.0000,0.50000


,,,,tir1,tir2
,lexeme,case,phono,,
40,avoir,pi3S,a,175.03125,87.515625
73,abaisser,inf,abɛse,0.25000,0.125000
128,abandonner,pi3S,abɑ̃dɔn,0.75000,0.375000
132,abandonner,inf,abɑ̃dɔne,3.31250,1.656250
133,abandonner,fi3S,abɑ̃dɔnəʁa,0.50000,0.250000


...

,,,,tir1,tir2
,lexeme,case,phono,,
40,avoir,pi3S,a,175.965474,87.982737
45,abaisser,ai3S,abɛsa,0.062500,0.031250
73,abaisser,inf,abɛse,0.031372,0.015686
108,abandonner,ai3S,abɑ̃dɔna,0.575203,0.287602
114,abandonner,pP,abɑ̃dɔnɑ̃,0.023926,0.011963


,,,,tir1,tir2
,lexeme,case,phono,,
40,avoir,pi3S,a,185.982737,92.991368
45,abaisser,ai3S,abɛsa,0.031250,0.015625
73,abaisser,inf,abɛse,0.015686,0.007843
108,abandonner,ai3S,abɑ̃dɔna,0.287602,0.143801
114,abandonner,pP,abɑ̃dɔnɑ̃,0.011963,0.005981


,,,,tir1,tir2
,lexeme,case,phono,,
40,avoir,pi3S,a,170.991368,85.495684
45,abaisser,ai3S,abɛsa,0.015625,0.007812
73,abaisser,inf,abɛse,0.007843,0.003922
102,abaisser,ppFS,abɛse,1.000000,0.500000
108,abandonner,ai3S,abɑ̃dɔna,0.143801,0.071900


,,,,tir1,tir2
,lexeme,case,phono,,
40,avoir,pi3S,a,172.495684,86.247842
45,abaisser,ai3S,abɛsa,0.007812,0.003906
102,abaisser,ppFS,abɛse,0.500000,0.250000
108,abandonner,ai3S,abɑ̃dɔna,0.071900,0.035950
128,abandonner,pi3S,abɑ̃dɔn,0.723756,0.361878


,,,,tir1,tir2
,lexeme,case,phono,,
40,avoir,pi3S,a,175.247842,87.623921
102,abaisser,ppFS,abɛse,0.250000,0.125000
108,abandonner,ai3S,abɑ̃dɔna,0.035950,0.017975
113,abandonner,ii3S,abɑ̃dɔnɛ,1.000000,0.500000
128,abandonner,pi3S,abɑ̃dɔn,0.361878,0.180939
